In [0]:
%pip install --upgrade great_expectations

%restart_python

import great_expectations as gx


![great_expectations.png](great_expectations.png)

In [0]:
%python

import pyspark.sql 
from delta.tables import DeltaTable
import great_expectations as gx
from great_expectations import expectations as expect

'''
   1. SET UP A GX ENVIRONMENT
'''
# Data context

contexto = gx.get_context()

'''
    2. CONNECT TO DATA
'''
# Data_Source - "db"
data_source = contexto.data_sources.add_spark(name="spark_datasource", persist=False)


# Data asset - "table"
df_dim_customers = spark.table("banking_gold.dim_customers")
df_dim_accounts = spark.table("banking_gold.dim_accounts")
df_transactions = spark.table("banking_gold.fact_transactions")

''' FUNCTION TO CREATE THE GREAT_EXPECTATIONS PIPELINE '''

def gx_pipeline_function (contexto, data_source, asset_name):
    data_asset = data_source.add_dataframe_asset(name=asset_name)

    # BATCH - Exact amount of data to evaluate - ENTIRE DF
    batch_definition = data_asset.add_batch_definition_whole_dataframe(name=f"batch_definition_{asset_name}")
    suite = contexto.suites.add(gx.ExpectationSuite(name=f"suite_{asset_name}"))

    return batch_definition, suite
    
''' Function calls -> data_asset - batch_definition - suites '''

# DIM_CUSTOMERS

batch_def_customers, suite_customers = gx_pipeline_function(contexto, data_source, "customers")

# DIM_ACCOUNTS
batch_def_accounts, suite_accounts = gx_pipeline_function(contexto, data_source, "accounts")

# FACT_TABLES
batch_def_trans, suite_trans = gx_pipeline_function(contexto, data_source, "trans")


# Add Expectations to Suites created simultaneously

# SUITE CUSTOMERS
suite_customers.add_expectation(
    expect.ExpectColumnValuesToNotBeNull(column="customer_sk", mostly=0.95)
)
suite_customers.add_expectation(
    expect.ExpectColumnValuesToBeUnique(column="customer_sk")
)

# SUITE ACCOUNTS
suite_accounts.add_expectation(
    expect.ExpectColumnValuesToNotBeNull(column="account_sk")
)
suite_accounts.add_expectation(
    expect.ExpectColumnValuesToBeUnique(column="account_sk")
)

# SUITE TRANSACTIONS
suite_trans.add_expectation(
    expect.ExpectColumnValuesToNotBeInSet(column="account_sk", value_set=["UNKNOWN"])
)
suite_trans.add_expectation(
    expect.ExpectColumnValuesToNotBeInSet(column="customer_sk", value_set=["UNKNOWN"])
)
suite_trans.add_expectation(
    expect.ExpectColumnValuesToBeUnique(column="transaction_id")
)
suite_trans.add_expectation(
    expect.ExpectColumnValuesToNotBeNull(column="transaction_id")
)
suite_trans.add_expectation(
    expect.ExpectColumnValuesToNotBeNull(column="transaction_date_id")
)
suite_trans.add_expectation(
    expect.ExpectColumnValuesToNotBeNull(column="amount")
)

# SAVE VALIDATIONS
validation_customers = gx.ValidationDefinition(data=batch_def_customers, 
                                               suite=suite_customers,
                                               name="validation_def_customers")

validation_accounts = gx.ValidationDefinition(data=batch_def_accounts, 
                                              suite=suite_accounts, 
                                              name="validation_def_accounts")

validation_transfer = gx.ValidationDefinition(data=batch_def_trans, 
                                              suite=suite_trans,
                                              name="validation_def_transfer")


validation_customers = contexto.validation_definitions.add(validation_customers)
validation_accounts = contexto.validation_definitions.add(validation_accounts)
validation_transfer = contexto.validation_definitions.add(validation_transfer)


## Execution
result_cust = validation_customers.run(batch_parameters={"dataframe": df_dim_customers})
result_acc = validation_accounts.run(batch_parameters={"dataframe": df_dim_accounts})
result_fact = validation_transfer.run(batch_parameters={"dataframe": df_transactions})

success = result_cust.success and result_acc.success and result_fact.success

# CHECK THAT IT WORKED
try:
    if success != True:
        raise Exception("One or more Great Expectations quality checks failed.")
except Exception as exp:
    print

ESTO ES LO QUE SUSTITUYE LA FUNCIÓN
- PODRÍA PLANTEAR ESTE PROCESO COMO UNA FUNCIÓN? CREACIÓN DE ASSET
data_asset_customers = data_source.add_dataframe_asset(name = "dim_customers_asset")
data_asset_accounts = data_source.add_dataframe_asset(name= "dim_accounts_asset")
data_asset_fact_trans = data_source.add_dataframe_asset(name = "fact_trans_asset")



- Batch Definition - instrucción o fórmula para coger los datos cuando se solicite un batch
batch_definition_cust = data_asset_customers.add_batch_definition_whole_dataframe(name="batch_whole_df_customers")
batch_definition_acc = data_asset_accounts.add_batch_definition_whole_dataframe(name="batch_whole_df_accounts")
batch_definition_facts = data_asset_fact_trans.add_batch_definition_whole_dataframe(name="batch_whole_df_facts_transaction")


'''
    3. Define Expectations
'''
- EXPECTATIONS SUITE -> lista expectativas que tienen que estar bien para pasar CHECK

- Creo la suite y la meto en el contexto
suite_customers = contexto.suites.add(ge.ExpectationSuite(name="dim_customers_suite"))
suite_accounts = contexto.suites.add(ge.ExpectationSuite(name="dim_accounts_suite"))
suite_trans = contexto.suites.add(ge.ExpectationSuite(name="fact_transaction_suite"))
